FUNCTION Init(puzzle_string, config):
    Load configuration and set domain_size, unassigned_cell, max_attempts flags
    Set strategy flags: use_mvc, use_lookahead, use_color (from args or config)
    
    IF puzzle_string is None:
        Load puzzle from file or config
    
    Create domain_size × domain_size grids for cells and original_clues
    SET backtrack_counter = 0
    Initialize grid from puzzle_string

FUNCTION initialize_grid(puzzle_string):
    Validate puzzle_string length equals domain_size²
    
    FOR i from 0 to domain_size² - 1:
        Calculate row = i ÷ domain_size
        Calculate col = i mod domain_size
        SET grid[row][col] = integer value of puzzle_string[i]
        SET original_clues[row][col] = (value ≠ unassigned_cell)

FUNCTION load_puzzle_from_file(config):
    Determine puzzle file path from config or defaults
    IF file exists, read puzzle string; ELSE load from config
    Parse and return puzzle parts

FUNCTION get_puzzle_parts(puzzle_string, config):
    Split by "_" into puzzle and solution (validate both are domain_size² length)
    Extract strategy flags from config
    RETURN puzzle, use_mvc, use_lookahead, use_color

FUNCTION is_safe(row, col, number):
    Check if number conflicts in row, column, or box:
    
    FOR i from 0 to domain_size - 1:
        IF grid[row][i] = number OR grid[i][col] = number:
            RETURN False
    
    Calculate box boundaries (box_size = √domain_size)
    FOR r from start_row to start_row + box_size - 1:
        FOR c from start_col to start_col + box_size - 1:
            IF grid[r][c] = number:
                RETURN False
    
    RETURN True

FUNCTION find_empty_cell():
    IF use_mvc is False:
        Simple search - return first empty cell:
        FOR row from 0 to domain_size - 1:
            FOR col from 0 to domain_size - 1:
                IF grid[row][col] = unassigned_cell:
                    RETURN (row, col)
    
    ELSE:
        MRV heuristic - return cell with fewest valid options:
        SET min_options = domain_size + 1
        SET best_cell = None
        
        FOR row from 0 to domain_size - 1:
            FOR col from 0 to domain_size - 1:
                IF grid[row][col] = unassigned_cell:
                    Count valid options for this cell:
                    SET options = 0
                    FOR num from 1 to domain_size:
                        IF is_safe(row, col, num):
                            INCREMENT options
                    
                    IF options < min_options:
                        SET min_options = options
                        SET best_cell = (row, col)
        
        RETURN best_cell
    
    RETURN None

FUNCTION check_lookahead(row, col):
    Check if any empty cell has zero valid options:
    
    FOR r from 0 to domain_size - 1:
        FOR c from 0 to domain_size - 1:
            IF grid[r][c] = unassigned_cell:
                Check if this cell has at least one valid number:
                FOR num from 1 to domain_size:
                    IF is_safe(r, c, num):
                        BREAK (this cell is ok)
                
                IF no valid number found:
                    RETURN True (failure detected)
    
    RETURN False

FUNCTION solve():
    IF use_max_attempts AND backtrack_counter ≥ max_attempts:
        RETURN False
    
    Find next empty cell (MRV or simple)
    IF no empty cell found:
        RETURN True (solved)
    
    Try each number in the cell:
    FOR number from 1 to domain_size:
        IF is_safe(row, col, number):
            SET grid[row][col] = number
            
            IF use_lookahead AND check_lookahead detects failure:
                SET grid[row][col] = unassigned_cell
                CONTINUE
            
            IF solve() returns True:
                RETURN True
            
            SET grid[row][col] = unassigned_cell (backtrack)
            INCREMENT backtrack_counter
    
    RETURN False

In [ ]:
import os
os.makedirs("config", exist_ok=True)

import sudoku_solver
print(dir(sudoku_solver))


['SudokuSolver', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'json', 'os']


In [ ]:
%%writefile config/config_solver.json
{
  "metadata": {
    "version": "0.0.1",
    "author": "Constantine E Hinds",
    "last_updated": "2025-11-28"
  },
  "domain_size": 9,
  "cell_size": 9,
  "use_max_attempts": false,
  "max_attempts": 10,
  "unassigned_cell":0,

  "use_mvc": true,
  "use_lookahead": false,
  "use_vizualization": false,

  "formats": {
    "use_custom_formatting": true,
    "font": {
      "use_custom_font": true,
      "family": "Arial",
      "size": 14
    },
    "cell_border": {
      "use_custom_border": true,
      "style": "1px solid black"
    },
    "alignment": {
      "use_custom_alignment": true,
      "horizontal": "center",
      "vertical": "middle"
    }
  },

  "visualizations": {
    "unsolved_puzzle": true,
    "solved_puzzle": true,
    "node_tree": false,
    "table_format": {
      "table": "white",
      "prefilled": "gray",
      "correct": "green",
      "incorrect": "red"
    }
  },

  "node_config": {
    "max_depth": 50,
    "display": {
      "show_node_id": true,
      "show_current_domain": true,
      "show_removed_values": true
    },
    "formatting": {
      "font": "Courier New",
      "font_size": 12,
      "node_color": "#333333",
      "domain_color": "#0055AA",
      "removed_values_color": "#AA0000",
      "background_color": "#F0F0F0",
      "border": "1px dashed #888888",
      "padding": "6px",
      "margin": "4px"
    },
    "backtracking": {
      "enabled": true,
      "label": "Backtrack",
      "font": "Courier New",
      "font_size": 12,
      "text_color": "#FFFFFF",
      "background_color": "#CC0000",
      "border": "1px solid #990000",
      "highlight_style": "italic"
    }
  },

  "manual_solving": {
    "enabled": true,
    "max_wrong_attempts": 3
  },

  "logging": {
    "enable_logging": true,
    "log_level": "INFO",
    "log_file": "solver.log"
  },

  "performance": {
    "max_recursion_depth": 10000,
    "timeout_seconds": 30
  },

  "strings": {
    "solution_intro": "The solution is as follows: {}",
    "puzzle_loaded": "Puzzle '{}' has been loaded.",
    "solving_started": "Solving puzzle '{}'. Please wait...",
    "solved_success": "Puzzle '{}' solved successfully!",
    "solved_failure": "Unable to solve puzzle '{}'. Please check the input.",
    "invalid_input": "The input provided is not valid. Please enter a valid puzzle string.",
    "load_error": "Error loading puzzle '{}'. File may be missing or corrupted.",
    "visualization_disabled": "Visualization is currently disabled in the configuration."
  },

  "puzzles": {
    "test_puzzle": "530070000600195000098000060800060003400803001700020006060000280000419005000080079_5346789126721953481983425678597761423426853791713924856961537284387419625245286179",
    "empty_puzzle": "000000000000000000000000000000000000000000000000000000000000000000000000000000000_123456789123456789123456789123456789123456789123456789123456789123456789123456789"
  }
}

Overwriting config/config_solver.json


In [ ]:
%%writefile sudoku_solver.py
import os
import json

class SudokuSolver:
    def __init__(self, puzzle_string=None, config=None, use_mvc=None, use_lookahead=None, use_color=None):
        if config is None:
            config = self.load_config()
        self.config = config

        # Initialize core configuration attributes early to be available for puzzle loading
        self.domain_size = config.get("domain_size", 9)
        self.unassigned_cell = config.get("unassigned_cell", 0)
        self.max_attempts = config.get("max_attempts", 10)
        self.use_max_attempts = config.get("use_max_attempts", False)

        # Determine strategy flags, prioritizing constructor arguments, then config
        self.use_mvc = use_mvc if use_mvc is not None else config.get("use_mvc", False)
        self.use_lookahead = use_lookahead if use_lookahead is not None else config.get("use_lookahead", False)
        self.use_color = use_color if use_color is not None else config.get("use_vizualization", False)


        if puzzle_string is None:
            # load_puzzle_from_file relies on self.domain_size, which is now initialized.
            # It also returns use_mvc, use_lookahead, use_color. The current logic
            # prioritizes constructor args, then config defaults for self.use_mvc etc.
            # For simplicity, we just extract the puzzle_string here.
            loaded_puzzle_string, _, _, _ = self.load_puzzle_from_file(config)
            puzzle_string = loaded_puzzle_string

        self.grid = [[self.unassigned_cell] * self.domain_size for _ in range(self.domain_size)]
        self.original_clues = [[False] * self.domain_size for _ in range(self.domain_size)]
        self.backtrack_counter = 0

        self.initialize_grid(puzzle_string)

    def load_config(self, path="config/config_solver.json"):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Configuration file '{path}' not found.")
        with open(path, "r") as config_file:
            return json.load(config_file)

    def initialize_grid(self, puzzle_string):
        if len(puzzle_string) != self.domain_size ** 2:
            raise ValueError(f"Invalid puzzle length. Must be {self.domain_size ** 2} characters.")
        for i in range(self.domain_size ** 2):
            row = i // self.domain_size
            col = i % self.domain_size
            value = int(puzzle_string[i])
            self.grid[row][col] = value
            self.original_clues[row][col] = value != self.unassigned_cell

    def load_puzzle_from_config(self, config, puzzle_name="test_puzzle"):
        puzzles = config.get("puzzles", {})
        puzzle = puzzles.get(puzzle_name)
        if not puzzle:
            raise ValueError(f"Puzzle '{puzzle_name}' not found in configuration.")
        return self.get_puzzle_parts(puzzle, config)

    def load_puzzle_from_file(self, config):
        puzzle_file = config.get("puzzle_file")
        if not puzzle_file:
            puzzle_dir = config.get("puzzle_dir", "puzzles")
            puzzle_name = config.get("puzzle_name", "test_puzzle")
            puzzle_file = os.path.join(puzzle_dir, f"{puzzle_name}.txt")

        if not os.path.exists(puzzle_file):
            return self.load_puzzle_from_config(config)

        with open(puzzle_file, "r") as f:
            puzzle_string = f.read().strip()

        return self.get_puzzle_parts(puzzle_string, config)

    def get_puzzle_parts(self, puzzle_string, config=None):
        parts = puzzle_string.split("_")
        if len(parts) != 2:
            raise ValueError("Puzzle must contain both puzzle and solution separated by '_'")
        # self.domain_size is now guaranteed to be set at this point
        if len(parts[0]) != self.domain_size ** 2 or len(parts[1]) != self.domain_size ** 2:
            raise ValueError(f"Invalid puzzle length. Must be {self.domain_size ** 2} characters.")

        puzzle = parts[0]
        solution = parts[1]
        # These values are returned, but currently not directly used to set self.attributes in __init__
        use_mvc_from_puzzle = config.get("use_mvc", False) if config else False
        use_lookahead_from_puzzle = config.get("use_lookahead", False) if config else False
        use_color_from_puzzle = config.get("use_vizualization", False) if config else False

        return puzzle, use_mvc_from_puzzle, use_lookahead_from_puzzle, use_color_from_puzzle

    def is_safe(self, row, col, number):
        # Check row and column
        for i in range(self.domain_size):
            if self.grid[row][i] == number or self.grid[i][col] == number:
                return False

        # Check 3x3 box
        box_size = int(self.domain_size ** 0.5)
        start_row = (row // box_size) * box_size
        start_col = (col // box_size) * box_size
        for r in range(start_row, start_row + box_size):
            for c in range(start_col, start_col + box_size):
                if self.grid[r][c] == number:
                    return False

        return True

    def find_empty_cell(self): #AI acknowledgement: struggled
        if not self.use_mvc:
            for row in range(self.domain_size):
                for col in range(self.domain_size):
                    if self.grid[row][col] == self.unassigned_cell:
                        return row, col
        else:
            min_options = self.domain_size + 1
            best_cell = None
            for row in range(self.domain_size):
                for col in range(self.domain_size):
                    if self.grid[row][col] == self.unassigned_cell:
                        options = sum(1 for num in range(1, self.domain_size + 1) if self.is_safe(row, col, num))
                        if options < min_options:
                            min_options = options
                            best_cell = (row, col)
            return best_cell

        return None

    def check_lookahead(self, row, col):
        for r in range(self.domain_size):
            for c in range(self.domain_size):
                if self.grid[r][c] == self.unassigned_cell:
                    if not any(self.is_safe(r, c, num) for num in range(1, self.domain_size + 1)):
                        return True
        return False

    def solve(self):
        if self.use_max_attempts and self.backtrack_counter >= self.max_attempts:
            return False

        next_cell = self.find_empty_cell()
        if not next_cell:
            return True  # Solved

        row, col = next_cell
        for number in range(1, self.domain_size + 1):
            if self.is_safe(row, col, number):
                self.grid[row][col] = number

                if self.use_lookahead and self.check_lookahead(row, col):
                    self.grid[row][col] = self.unassigned_cell
                    continue

                if self.solve():
                    return True

                self.grid[row][col] = self.unassigned_cell
                self.backtrack_counter += 1

        return False

Overwriting sudoku_solver.py


In [ ]:
from sudoku_solver import SudokuSolver

def print_grid(grid):
    for row in grid:
        print(" ".join(str(num) if num != 0 else "." for num in row))

# Run the solver
try:
    solver = SudokuSolver()  # Automatically loads config and puzzle
    print("Initial Puzzle:")
    print_grid(solver.grid)

    if solver.solve():
        print("\nSolved Puzzle:")
        print_grid(solver.grid)
        print(f"\nBacktracks: {solver.backtrack_counter}")
    else:
        print("\nNo solution found.")
        print(f"Backtracks attempted: {solver.backtrack_counter}")

except Exception as e:
    print(f"Error: {e}")

Error: 'SudokuSolver' object has no attribute 'domain_size'
